In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mdibrahimsani/markopolo/takapay_sample_data.json
/kaggle/input/datasets/mdibrahimsani/markopolo/takapay_sample_data.csv


In [2]:
import pandas as pd
 
# ---- 1. Load ----
DATA_PATH = "/kaggle/input/datasets/mdibrahimsani/markopolo/takapay_sample_data.csv"
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows")
 
# ---- 2. Basic type cleanup ----
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["sentiment"] = df["sentiment"].str.lower().str.strip()
df["topic"] = df["topic"].str.lower().str.strip()
 
# ---- 3. Reconcile sentiment label vs sentiment_score ----
# Some rows have a sentiment label that disagrees with the numeric score.
# Decision: trust sentiment_score as the source of truth (it's more granular),
# and derive a consistent label from it. Keep original label for reference.
def score_to_band(score):
    if score >= 60:
        return "positive"
    elif score <= 40:
        return "negative"
    else:
        return "neutral"
 
df["sentiment_original"] = df["sentiment"]
df["sentiment"] = df["sentiment_score"].apply(score_to_band)
 
mismatches = (df["sentiment"] != df["sentiment_original"]).sum()
print(f"Reconciled {mismatches} sentiment label/score mismatches "
      f"(kept sentiment_score as source of truth)")
 
# ---- 4. Flag duplicate posts (same text, different id/author) ----
# Likely bot / spam / copy-paste activity. We don't drop them (they're still
# real signal about volume), but we flag them so the dashboard can show a
# "possible duplicate/spam" note.
df["is_duplicate_text"] = df.duplicated(subset=["text"], keep=False)
print(f"Flagged {df['is_duplicate_text'].sum()} rows involved in duplicate text groups")
 
# ---- 5. Off-topic posts ----
# topic == 'off_topic' means the post doesn't actually discuss TakaPay
# (traffic, food, exams, etc.) even though brand_mention=True in the raw data.
# Decision: exclude these from sentiment/topic aggregates by default,
# but keep them in the raw table and show the count transparently.
df["include_in_analysis"] = df["topic"] != "off_topic"
excluded = (~df["include_in_analysis"]).sum()
print(f"Excluding {excluded} off-topic rows from core sentiment/topic aggregates")
 
# ---- 6. Save cleaned dataset ----
df.to_csv("takapay_cleaned.csv", index=False)
 
# ---- 7. Build summary JSON for the dashboard ----
analysis_df = df[df["include_in_analysis"]]
 
summary = {
    "total_records": int(len(df)),
    "analyzed_records": int(len(analysis_df)),
    "excluded_off_topic": int(excluded),
    "duplicate_flagged": int(df["is_duplicate_text"].sum()),
    "sentiment_breakdown": analysis_df["sentiment"].value_counts().to_dict(),
    "topic_breakdown": analysis_df["topic"].value_counts().to_dict(),
    "platform_breakdown": analysis_df["platform"].value_counts().to_dict(),
    "avg_sentiment_score": round(analysis_df["sentiment_score"].mean(), 1),
    "competitor_mentions": int((analysis_df["topic"] == "competitor").sum()),
}
 
import json
with open("takapay_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
 
print("\nSaved: takapay_cleaned.csv, takapay_summary.json")
print(json.dumps(summary, indent=2, ensure_ascii=False))
 

Loaded 660 rows
Reconciled 24 sentiment label/score mismatches (kept sentiment_score as source of truth)
Flagged 20 rows involved in duplicate text groups
Excluding 61 off-topic rows from core sentiment/topic aggregates

Saved: takapay_cleaned.csv, takapay_summary.json
{
  "total_records": 660,
  "analyzed_records": 599,
  "excluded_off_topic": 61,
  "duplicate_flagged": 20,
  "sentiment_breakdown": {
    "negative": 338,
    "positive": 215,
    "neutral": 46
  },
  "topic_breakdown": {
    "failed_transaction": 220,
    "competitor": 81,
    "cashback_offer": 63,
    "send_money": 50,
    "recharge": 50,
    "charges_fees": 30,
    "agent_network": 26,
    "bill_payment": 24,
    "feature_query": 19,
    "customer_care": 13,
    "login_otp": 10,
    "app_crash": 9,
    "app_experience": 2,
    "product_news": 2
  },
  "platform_breakdown": {
    "Facebook": 200,
    "Reddit": 76,
    "News/Media": 72,
    "Instagram": 71,
    "YouTube": 65,
    "TikTok": 58,
    "Twitter": 57
  },
  